In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer, Embedding, Dense, LayerNormalization, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re
import string

In [2]:
df_passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
df_passages.head()

,passage
id,
9797,New data on viruses isolated from patients wit...
11906,We describe an improved method for detecting d...
16083,We have studied the effects of curare on respo...
23188,Kinetic and electrophoretic properties of 230-...
23469,Male Wistar specific-pathogen-free rats aged 2...


In [4]:
df_test.head()

,question,answer,relevant_passage_ids
id,,,
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011..."
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787..."
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004..."
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 21622663, 22707570, 22955988, 24285..."
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 23827649, 21618594, 23835909, 24265..."


In [6]:
passage_dict = pd.Series(df_passages['passage'].values, index=df_passages.index).to_dict()

In [7]:
def map_passages(id_list):
    if isinstance(id_list, list):
        return " ".join(passage_dict.get(pid, "") for pid in id_list)
    return ""


In [8]:
df_test['relevant_passages_text'] = df_test['relevant_passage_ids'].apply(map_passages)

In [9]:
# 3. Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [10]:
df_test['question_clean'] = df_test['question'].astype(str).apply(clean_text)
df_test['relevant_passages_clean'] = df_test['relevant_passages_text'].astype(str).apply(clean_text)

In [11]:
# Combine all text to fit tokenizer
texts = df_test['question_clean'].tolist() + df_test['relevant_passages_clean'].tolist()


In [12]:
# 4. Tokenizer and sequences
# ---------------------------
max_vocab = 20000
max_len = 128

tokenizer = Tokenizer(num_words=max_vocab, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
word_index = tokenizer.word_index
vocab_size = min(max_vocab, len(word_index)) + 1


In [13]:
# Convert texts to padded sequences
question_seqs = tokenizer.texts_to_sequences(df_test['question_clean'])
passage_seqs = tokenizer.texts_to_sequences(df_test['relevant_passages_clean'])

question_padded = pad_sequences(question_seqs, maxlen=max_len, padding='post')
passage_padded = pad_sequences(passage_seqs, maxlen=max_len, padding='post')


In [14]:
print(f"Vocab size: {vocab_size}")


Vocab size: 7211


In [16]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2025-07-20 06:20:26--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-07-20 06:20:26--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-07-20 06:20:27--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [18]:
# 5. Load GloVe embeddings and build embedding matrix
# ---------------------------
embedding_dim = 100
embedding_index = {}

glove_path = 'glove.6B.100d.txt'  # Make sure this file is present

with open(glove_path, encoding='utf8') as f:
    for line in f:
        values = line.strip().split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    if i >= vocab_size:
        continue
    vec = embedding_index.get(word)
    if vec is not None:
        embedding_matrix[i] = vec

In [19]:
# 6. Define Relative Positional Encoding Layer
# ---------------------------
class RelativePositionalEncoding(Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        self.rel_pos_emb = Embedding(input_dim=2*max_len - 1, output_dim=d_model)

    def call(self, seq_len):
        range_vec = tf.range(seq_len)
        rel_pos = range_vec[None, :] - range_vec[:, None] + self.max_len - 1
        return self.rel_pos_emb(rel_pos)

In [20]:
# 7. Transformer Encoder Layer
# ---------------------------
class TransformerEncoderLayer(Layer):
    def __init__(self, d_model, num_heads, dff, max_len):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads

        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            Dense(dff, activation='relu'),
            Dense(d_model)
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(0.1)
        self.dropout2 = tf.keras.layers.Dropout(0.1)

        self.rel_pos_encoding = RelativePositionalEncoding(max_len, d_model)
        self.max_len = max_len

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]

        rel_pos_emb = self.rel_pos_encoding(seq_len)  # (seq_len, seq_len, d_model)

        attn_output = self.mha(query=x, key=x, value=x, training=training)

        # Approximate integration of relative positional embeddings by adding mean vector per position
        rel_pos_mean = tf.reduce_mean(rel_pos_emb, axis=1)  # (seq_len, d_model)
        attn_output += rel_pos_mean[None, :, :]

        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.norm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.norm2(out1 + ffn_output)

        return out2

In [21]:
# 8. Build Model Example (e.g. on questions)
# ---------------------------
d_model = embedding_dim
num_heads = 4
dff = 256

inputs = Input(shape=(max_len,))
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=max_len,
    trainable=False
)(inputs)

transformer_encoder = TransformerEncoderLayer(d_model, num_heads, dff, max_len)(embedding_layer)

model = Model(inputs=inputs, outputs=transformer_encoder)
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 128, 100)       │       721,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_layer       │ (None, 128, 100)       │       238,756 │
│ (TransformerEncoderLayer)       │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 959,856 (3.66 MB)

 Trainable params: 238,756 (932.64 KB)

 Non-trainable params: 721,100 (2.75 MB)

In [22]:
# 9. Run a test forward pass on question sequences
# ---------------------------
output = model(question_padded[:5])
print("Output shape:", output.shape)

Output shape: (5, 128, 100)


 What i got:

BioASQ data loaded and joined

Cleaned text ready for tokenizer

Tokenizer fitted on questions + passages

Embedding matrix built from GloVe

Transformer encoder with relative positional encoding implemented

Model summary and test forward pass output shape printed